# Notebook 09A — NCCS Source Completeness and Feature Integrity Audit

**Purpose:** Audit the existing 2010–2024 NCCS longitudinal feature matrix against raw SUMMARY and GOVERNANCE Parquet data. This notebook **does not modify** source files, features, or trained models.

Checks: source coverage; NULL-to-zero conversion; known-value mismatches; feature-level NULL patterns; derived governance flags; preprocessing configuration; and an optional small sample of discrepancies. This is an audit, **not** model optimization (planned for Notebook 09B).

**Important:** The six raw-to-final fields previously showed zero NULL-to-zero conversions and zero value mismatches. The notebook independently reproduces and extends that check. Counts of missing records and missing field values are distinct. It does not assume missing source values should be zero.

In [ ]:
from pathlib import Path
import json
import duckdb
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SOURCE_DIR = ROOT / "data/external/nccs/efile_v3_1"
FEATURE_FILE = ROOT / "data/processed/nccs_longitudinal_features_v1.parquet"
OUT = ROOT / "data/processed/notebook09a"
OUT.mkdir(parents=True, exist_ok=True)

summary_files = sorted(SOURCE_DIR.rglob("*SUMMARY*.parquet"))
governance_files = sorted(SOURCE_DIR.rglob("*GOVERNANCE*.parquet"))
assert FEATURE_FILE.is_file(), f"Feature file not found: {FEATURE_FILE}"
assert summary_files and governance_files, f"Missing source files under {SOURCE_DIR}"
print("Project root:", ROOT)
print("SUMMARY files:", len(summary_files))
print("GOVERNANCE files:", len(governance_files))
print("Feature file:", FEATURE_FILE)
print("Output folder:", OUT)

## 1. Inspect schemas before assuming column names

The source column names below match the previously successful audit. This cell verifies that the required columns exist before scanning the data.

In [ ]:
con = duckdb.connect()
con.execute("SET threads = 4")
summary_paths = [p.as_posix() for p in summary_files]
governance_paths = [p.as_posix() for p in governance_files]

raw_mappings = {
    "total_assets": ("summary", "F9_01_NAFB_ASSET_TOT_EOY"),
    "total_liabilities": ("summary", "F9_01_NAFB_LIAB_TOT_EOY"),
    "total_revenue": ("summary", "F9_01_REV_TOT_CY"),
    "total_expenses": ("summary", "F9_01_EXP_TOT_CY"),
    "voting_members": ("governance", "F9_06_GVRN_NUM_VOTING_MEMB"),
    "independent_members": ("governance", "F9_06_GVRN_NUM_VOTING_MEMB_IND"),
}

def column_names(paths):
    return set(con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?, union_by_name=true)", [paths]
    ).df()["column_name"])

schema = {
    "summary": column_names(summary_paths),
    "governance": column_names(governance_paths),
    "features": column_names([FEATURE_FILE.as_posix()]),
}
required = {"OBJECTID"}
issues = []
for field, (source, raw_col) in raw_mappings.items():
    if raw_col not in schema[source]:
        issues.append(f"{source} missing {raw_col}")
    if field not in schema["features"]:
        issues.append(f"features missing {field}")
for name in ("summary", "governance", "features"):
    if "OBJECTID" not in schema[name]:
        issues.append(f"{name} missing OBJECTID")
assert not issues, "Schema mismatch:\n" + "\n".join(issues)
print("Required columns found in all three datasets.")
print("Final feature columns:", len(schema["features"]))

## 2. Define source and feature views

The source views use `TRY_CAST` to avoid treating malformed numeric strings as valid numbers. A separate casting diagnostic below checks whether this occurs. `OBJECTID` is the audit join key.

In [ ]:
def qident(name):
    return '"' + name.replace('"', '""') + '"'

def source_select(source):
    parts = ["OBJECTID"]
    for field, (dataset, raw_col) in raw_mappings.items():
        if dataset == source:
            parts.append(f"TRY_CAST({qident(raw_col)} AS DOUBLE) AS {qident(field)}")
    return ",\n        ".join(parts)

con.execute(f"""
    CREATE OR REPLACE VIEW raw_summary AS
    SELECT {source_select('summary')}
    FROM read_parquet({repr(summary_paths)}, union_by_name=true)
""")
con.execute(f"""
    CREATE OR REPLACE VIEW raw_governance AS
    SELECT {source_select('governance')}
    FROM read_parquet({repr(governance_paths)}, union_by_name=true)
""")
con.execute(f"""
    CREATE OR REPLACE VIEW final_features AS
    SELECT * FROM read_parquet('{FEATURE_FILE.as_posix().replace("'", "''")}')
""")
print("Views created.")

## 3. Join-key integrity and source coverage

If raw source tables have duplicate `OBJECTID` keys, a naive join can inflate audit counts. Verify uniqueness first and stop if duplicates are found. The feature dataset should have one row per selected filing.

In [ ]:
key_checks = []
for name in ("final_features", "raw_summary", "raw_governance"):
    row = con.execute(f"""
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT OBJECTID) AS unique_objectids,
               COUNT(*) FILTER (WHERE OBJECTID IS NULL) AS null_objectids
        FROM {name}
    """).df().iloc[0].to_dict()
    row["dataset"] = name
    row["duplicate_rows_by_key"] = row["rows"] - row["unique_objectids"] - row["null_objectids"]
    key_checks.append(row)
key_audit = pd.DataFrame(key_checks)[
    ["dataset", "rows", "unique_objectids", "null_objectids", "duplicate_rows_by_key"]
]
display(key_audit)
key_audit.to_csv(OUT / "01_join_key_integrity.csv", index=False)
assert (key_audit["null_objectids"] == 0).all(), "NULL OBJECTID detected; inspect before joining"
assert (key_audit["duplicate_rows_by_key"] == 0).all(), "Duplicate OBJECTID detected; inspect before joining"
print("Join keys are unique and non-null.")

## 4. Raw-to-final value preservation audit

For each field, compare the raw source value with the selected feature value. `source_null_final_zero` counts missing source records **and** NULL source fields that became zero. `value_mismatch` counts changed known values. Missing source records are also reported separately.

In [ ]:
audit_rows = []

for field, (source, _) in raw_mappings.items():
    source_view = "raw_summary" if source == "summary" else "raw_governance"

    result = con.execute(f"""
        SELECT
            COUNT(*) AS total_filings,

            COUNT(*) FILTER (
                WHERE r.OBJECTID IS NULL
            ) AS missing_source_record,

            COUNT(*) FILTER (
                WHERE r.OBJECTID IS NOT NULL
                  AND r.{field} IS NULL
            ) AS source_null,

            COUNT(*) FILTER (
                WHERE r.{field} IS NULL
                  AND f.{field} = 0
            ) AS source_null_final_zero,

            COUNT(*) FILTER (
                WHERE r.{field} IS NOT NULL
                  AND f.{field} IS NULL
            ) AS source_known_final_null,

            COUNT(*) FILTER (
                WHERE r.{field} IS NOT NULL
                  AND f.{field} IS NOT NULL
                  AND ABS(
                      r.{field} - TRY_CAST(f.{field} AS DOUBLE)
                  ) > (
                      1e-6 + 1e-9 * GREATEST(
                          ABS(r.{field}),
                          ABS(TRY_CAST(f.{field} AS DOUBLE))
                      )
                  )
            ) AS value_mismatch,

            COUNT(*) FILTER (
                WHERE f.{field} IS NULL
            ) AS final_null,

            COUNT(*) FILTER (
                WHERE f.{field} = 0
            ) AS final_zero

        FROM final_features f
        LEFT JOIN {source_view} r
            USING (OBJECTID)
    """).df().iloc[0].to_dict()

    audit_rows.append({
        "feature": field,
        "source": source,
        **result
    })

source_audit = pd.DataFrame(audit_rows)

display(source_audit)

source_audit.to_csv(
    OUT / "02_source_value_preservation.csv",
    index=False
)

problem_cols = [
    "source_null_final_zero",
    "source_known_final_null",
    "value_mismatch"
]

print(
    "Total discrepancies:",
    int(source_audit[problem_cols].sum().sum())
)

print(
    "NOTE: A zero discrepancy count does not mean source data is complete."
)

## 5. Numeric casting diagnostics

A failed `TRY_CAST` looks like a NULL. This check distinguishes genuinely missing values from non-empty source strings that cannot be interpreted as numbers.

In [ ]:
casting_rows = []
for field, (source, raw_col) in raw_mappings.items():
    paths = summary_paths if source == "summary" else governance_paths
    res = con.execute(f"""
        SELECT COUNT(*) AS raw_rows,
               COUNT(*) FILTER (
                   WHERE {qident(raw_col)} IS NOT NULL
                     AND NULLIF(TRIM(CAST({qident(raw_col)} AS VARCHAR)), '') IS NOT NULL
                     AND TRY_CAST({qident(raw_col)} AS DOUBLE) IS NULL
               ) AS nonempty_cast_failures
        FROM read_parquet({repr(paths)}, union_by_name=true)
    """).df().iloc[0].to_dict()
    casting_rows.append({"feature": field, **res})
casting_audit = pd.DataFrame(casting_rows)
display(casting_audit)
casting_audit.to_csv(OUT / "03_casting_diagnostics.csv", index=False)

## 6. Missingness in the final feature matrix, by year

This checks the stored Parquet values, not imputed model inputs. It resolves whether the apparent 0% missingness in Notebook 09 reflects preprocessing or a different sample.

In [ ]:
fields = list(raw_mappings)
exprs = ",\n".join(
    f"ROUND(100.0 * COUNT(*) FILTER (WHERE {field} IS NULL) / NULLIF(COUNT(*), 0), 3) AS {field}_missing_pct"
    for field in fields
)
yearly = con.execute(f"""
    SELECT tax_year, COUNT(*) AS filings, {exprs}
    FROM final_features
    GROUP BY tax_year
    ORDER BY tax_year
""").df()
display(yearly)
yearly.to_csv(OUT / "04_yearly_feature_missingness.csv", index=False)

label_expr = "fraud_label" if "fraud_label" in schema["features"] else None
if label_expr:
    labeled_missing = con.execute(f"""
        SELECT CASE WHEN tax_year <= 2020 THEN 'train_2010_2020'
                    WHEN tax_year <= 2022 THEN 'validation_2021_2022'
                    ELSE 'test_2023_2024' END AS period,
               COUNT(*) AS labeled_rows, {exprs}
        FROM final_features
        WHERE tax_year BETWEEN 2010 AND 2024 AND fraud_label IN (0, 1)
        GROUP BY 1 ORDER BY 1
    """).df()
    display(labeled_missing)
    labeled_missing.to_csv(OUT / "05_labeled_period_missingness.csv", index=False)

## 7. Derived governance feature checks

These checks identify potentially inconsistent flags. They do **not** impose an assumed formula for `flag_small_board` or `flag_minority_independent_board` because their exact thresholds should be confirmed in Notebook 04/04A/08 before asserting errors. For `pct_independent_board`, compare against the ratio only when the denominator is positive.

In [ ]:
feature_cols = schema["features"]
checks = []
if "pct_independent_board" in feature_cols:
    checks.append(("ratio_differs_when_positive_board", """
        voting_members > 0 AND independent_members IS NOT NULL
        AND pct_independent_board IS NOT NULL
        AND ABS(pct_independent_board - independent_members / voting_members) > 1e-5
    """))
    checks.append(("ratio_present_with_missing_board", """
        pct_independent_board IS NOT NULL
        AND (voting_members IS NULL OR independent_members IS NULL)
    """))
if "flag_board_independence_drop" in feature_cols:
    checks.append(("board_drop_flag_outside_0_1", """
        flag_board_independence_drop IS NOT NULL
        AND flag_board_independence_drop NOT IN (0, 1)
    """))
for flag in ("flag_small_board", "flag_minority_independent_board"):
    if flag in feature_cols:
        checks.append((f"{flag}_outside_0_1", f"{flag} IS NOT NULL AND {flag} NOT IN (0, 1)"))

flag_rows = []
for name, condition in checks:
    count = con.execute(f"SELECT COUNT(*) FROM final_features WHERE {condition}").fetchone()[0]
    flag_rows.append({"check": name, "count": count})
flag_audit = pd.DataFrame(flag_rows)
display(flag_audit)
flag_audit.to_csv(OUT / "06_governance_derived_checks.csv", index=False)
print("Review Notebook 04/04A/08 flag definitions before interpreting any flagged rows.")

## 8. Inspect saved Notebook 09 model preprocessing (if available)

The selected model is saved separately. This section reports its structure; it does not assume that the pipeline includes missingness indicators or that the feature table itself was imputed.

In [ ]:
model_path = ROOT / "data/processed/notebook09/selected_model.joblib"
manifest_path = ROOT / "data/processed/notebook09/model_manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print("Model manifest:")
    print(json.dumps(manifest, indent=2, default=str)[:12000])
else:
    print("No model manifest found; skip manifest inspection.")

if model_path.exists():
    import joblib
    model = joblib.load(model_path)  # Only load model artifacts you created and trust.
    print("\nSaved model type:", type(model).__name__)
    if hasattr(model, "steps"):
        print("Pipeline steps:", [(name, type(step).__name__) for name, step in model.steps])
        for name, step in model.steps:
            if hasattr(step, "add_indicator"):
                print(f"{name}.add_indicator = {step.add_indicator}")
else:
    print("No saved model found; skip model inspection.")

## 9. Optional: examples of discrepancies

Only run this if the source audit found any `source_null_final_zero` or `value_mismatch` rows. It limits output to 20 examples per problematic feature.

In [ ]:
bad_fields = source_audit.loc[
    (source_audit["source_null_final_zero"] > 0) |
    (source_audit["source_known_final_null"] > 0) |
    (source_audit["value_mismatch"] > 0), "feature"
].tolist()
if not bad_fields:
    print("No raw-to-final discrepancies detected in the six audited fields.")
else:
    for field in bad_fields:
        source = raw_mappings[field][0]
        view = "raw_summary" if source == "summary" else "raw_governance"
        sample = con.execute(f"""
            SELECT f.OBJECTID, f.tax_year, r.{field} AS source_value,
                   f.{field} AS final_value
            FROM final_features f LEFT JOIN {view} r USING (OBJECTID)
            WHERE (r.{field} IS NULL AND f.{field} = 0)
               OR (r.{field} IS NOT NULL AND f.{field} IS NULL)
               OR (r.{field} IS NOT NULL AND f.{field} IS NOT NULL
                   AND NOT isclose(r.{field}, TRY_CAST(f.{field} AS DOUBLE), rel_tol := 1e-10, abs_tol := 1e-8))
            LIMIT 20
        """).df()
        print("Discrepancy examples:", field)
        display(sample)

## 10. Audit summary and next actions

**Interpretation guide:**

- Zero `source_null_final_zero`, `source_known_final_null`, and `value_mismatch` counts support preservation of the six audited source values.
- `missing_source_record` is not the same as a source NULL; neither is automatically a processing error.
- Nonzero casting failures require inspecting raw strings and any type conversions.
- Nonzero governance checks require reviewing the actual feature definition before changing data.
- The model preprocessing inspection helps determine whether Notebook 09's missingness table was computed before or after imputation.

**Next:** Preserve Notebook 09 unchanged. After resolving any issues, create Notebook 09B for validation-only model optimization (2010–2020 training, 2021–2022 validation; do not tune to 2023–2024 holdout).

In [ ]:
summary = {
    "feature_file": str(FEATURE_FILE),
    "summary_source_files": len(summary_files),
    "governance_source_files": len(governance_files),
    "source_to_zero_errors": int(source_audit["source_null_final_zero"].sum()),
    "known_to_null_errors": int(source_audit["source_known_final_null"].sum()),
    "known_value_mismatches": int(source_audit["value_mismatch"].sum()),
    "nonempty_numeric_cast_failures": int(casting_audit["nonempty_cast_failures"].sum()),
    "derived_governance_check_counts": {str(r["check"]): int(r["count"]) for r in flag_rows},
    "outputs_folder": str(OUT),
}
(OUT / "audit_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))